<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Demo - Real-time Deployment with Model Serving

## Overview
Once a model has been evaluated, the final step is to make it available for real-time use. This lesson first explains the deployment paradigms for generative AI and how Databricks Model Serving and the Unity Gateway support production deployment, then walks through deploying the registered RAG model to a real-time serving endpoint with the Agent Bricks Custom Agents and collecting human feedback through the Review App.

## Learning Objectives
By the end of this demo, you will be able to:
- Describe the batch, streaming, and real-time deployment paradigms for generative AI.
- Explain how Databricks Model Serving deploys models as production endpoints.
- Describe how the Unity Gateway provides centralized governance for Gen AI deployments.
- Deploy a model to a real-time endpoint using the Agent Bricks Custom Agents.
- Interact with the deployed model and collect human feedback using the Review App.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, ML Environment Version 5 (ML v5)</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless compute with the ML v5 environment</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

To select the **ML v5** environment:

1. Click the **Environment** button on the right sidebar
2. Open the **Environment version** dropdown
3. Select **ML v5**

If you do not see **Serverless** available, contact your workspace administrator.

## A. Deployment Paradigms

<div class="dp">
  <style>
    @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&display=swap');
    .dp{font-family:'DM Sans','Segoe UI',Arial,sans-serif;max-width:1040px;margin:14px auto 6px;color:#1B3139;}
    .dp *{box-sizing:border-box;}
    .dp-bars{display:flex;flex-direction:column;gap:10px;margin-bottom:18px;}
    .dp-bar{position:relative;height:38px;border-radius:8px 0 0 8px;display:flex;align-items:center;justify-content:center;color:#fff;font-weight:800;font-size:14px;}
    .dp-bar::after{content:'';position:absolute;right:-19px;top:0;border-top:19px solid transparent;border-bottom:19px solid transparent;border-left:20px solid var(--c);}
    .dp-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:0;border-radius:14px;overflow:hidden;box-shadow:0 6px 16px rgba(27,49,57,.09);}
    .dp-card{padding:22px 20px;border-top:5px solid var(--acc);background:var(--bg);}
    .dp-card h4{margin:0 0 10px;font-size:16px;font-weight:800;color:var(--acc);}
    .dp-card p{margin:0;font-size:13.5px;line-height:1.55;color:#3a4d54;}
    .dp-note{margin-top:16px;background:#FFF6F4;border:1px solid #FADAD3;border-left:5px solid #FF5F46;border-radius:12px;padding:13px 16px;font-size:13.5px;color:#3a4d54;}
    @media(max-width:640px){.dp-grid{grid-template-columns:1fr;}}
  </style>
  <div class="dp-bars">
    <div class="dp-bar" style="--c:#FF5F46;background:#FF5F46;margin-right:22px;">Latency decreases &rarr;</div>
    <div class="dp-bar" style="--c:#00A972;background:#00A972;margin-right:22px;">Throughput decreases* &rarr;</div>
  </div>
  <div class="dp-grid">
    <div class="dp-card" style="--acc:#00A972;--bg:#EAF7F0;"><h4>Batch</h4><p>Generate and store completions on a table of (text) inputs / prompts.</p></div>
    <div class="dp-card" style="--acc:#4299E0;--bg:#EAF2FB;"><h4>Streaming</h4><p>Generate and store completions on micro-batches of inputs / prompts as they are processed.</p></div>
    <div class="dp-card" style="--acc:#FF5F46;--bg:#FFF1EE;"><h4>Real-time</h4><p>Generate completions asynchronously in real time on an individual input / prompt.</p></div>
  </div>
  <div class="dp-note"><b>Note:</b> Edge (on-device) deployment is challenging with large language models due to space requirements.</div>
</div>

## B. Challenges with Deploying Real-time AI

<div class="ch">
  <style>
    @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&display=swap');
    .ch{font-family:'DM Sans','Segoe UI',Arial,sans-serif;max-width:1040px;margin:14px auto 6px;color:#1B3139;}
    .ch *{box-sizing:border-box;}
    .ch-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:18px;}
    .ch-card{background:#fff;border-radius:15px;border-top:5px solid var(--acc);box-shadow:0 6px 16px rgba(27,49,57,.08);padding:22px 20px 18px;}
    .ch-badge{position:relative;width:52px;height:52px;border-radius:14px;background:var(--tint);color:var(--acc);display:inline-flex;align-items:center;justify-content:center;margin-bottom:13px;}
    .ch-badge svg{width:28px;height:28px;}
    .ch-pip{position:absolute;top:-5px;right:-5px;width:22px;height:22px;border-radius:50%;background:#FF3621;color:#fff;font-weight:800;font-size:13px;display:inline-flex;align-items:center;justify-content:center;border:3px solid #fff;box-shadow:0 2px 5px rgba(27,49,57,.2);}
    .ch-card h4{margin:0 0 9px;font-size:15.5px;font-weight:800;color:#0b2026;}
    .ch-card ul{list-style:none;margin:0;padding:0;}
    .ch-card li{position:relative;padding-left:18px;font-size:13px;line-height:1.5;color:#3a4d54;margin-bottom:8px;}
    .ch-card li::before{content:'';position:absolute;left:2px;top:7px;width:7px;height:7px;border-radius:2px;transform:rotate(45deg);background:var(--acc);}
    @media(max-width:720px){.ch-grid{grid-template-columns:1fr;}}
  </style>
  <div class="ch-grid">
    <div class="ch-card" style="--acc:#FF5F46;--tint:rgba(255,95,70,.12);"><div class="ch-badge"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><rect x="3" y="4" width="18" height="6.2" rx="1.6"/><rect x="3" y="13.8" width="18" height="6.2" rx="1.6"/><circle cx="7" cy="7.1" r="0.7" fill="currentColor" stroke="none"/><circle cx="7" cy="16.9" r="0.7" fill="currentColor" stroke="none"/><line x1="11" y1="7.1" x2="17.5" y2="7.1"/><line x1="11" y1="16.9" x2="17.5" y2="16.9"/></svg><span class="ch-pip">!</span></div><h4>Infrastructure is hard</h4><ul><li>Real-time AI needs fast, scalable serving infrastructure &mdash; costly to build and maintain.</li></ul></div>
    <div class="ch-card" style="--acc:#FF5F46;--tint:rgba(255,95,70,.12);"><div class="ch-badge"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M10.4 3.6a1.9 1.9 0 013.7 0c0 .6-.3 1.1-.3 1.6 0 .5.4.8.9.8h2.4c.6 0 1 .5 1 1v2.4c0 .5.3.9.8.9.5 0 1-.3 1.6-.3a1.9 1.9 0 010 3.7c-.6 0-1.1-.3-1.6-.3-.5 0-.8.4-.8.9v3.5c0 .6-.5 1-1 1h-3.5c-.5 0-.9-.4-.9-.9 0-.5.3-1 .3-1.6a1.9 1.9 0 00-3.7 0c0 .6.3 1.1.3 1.6 0 .5-.4.9-.9.9H4.6c-.6 0-1-.4-1-1v-3.5c0-.5.4-.8.9-.8.5 0 1 .3 1.6.3a1.9 1.9 0 000-3.7c-.6 0-1.1.3-1.6.3-.5 0-.9-.4-.9-.9V8c0-.5.4-1 1-1H8c.5 0 .8-.3.8-.8 0-.5-.3-1-.3-1.6z"/></svg><span class="ch-pip">!</span></div><h4>Deploying real-time models needs disparate tools</h4><ul><li>Data teams use diverse tools to develop models.</li><li>Separate platforms for data, LLMs and serving add complexity and cost.</li></ul></div>
    <div class="ch-card" style="--acc:#FF5F46;--tint:rgba(255,95,70,.12);"><div class="ch-badge"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M12 4 2 9l10 5 10-5-10-5Z"/><path d="M6 11.5v4.2c0 1.3 2.7 2.8 6 2.8s6-1.5 6-2.8v-4.2"/><line x1="22" y1="9" x2="22" y2="14.5"/></svg><span class="ch-pip">!</span></div><h4>Operating production AI requires expert resources</h4><ul><li>Steep learning curve of deployment tools.</li><li>Limited engineering resources bottleneck deployment and scaling.</li></ul></div>
  </div>
</div>

## C. Databricks Model Serving

<div class="ms">
  <style>
    @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&display=swap');
    .ms{font-family:'DM Sans','Segoe UI',Arial,sans-serif;max-width:1040px;margin:14px auto 6px;color:#1B3139;}
    .ms *{box-sizing:border-box;}
    .ms-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:18px;}
    .ms-card{background:#fff;border-radius:15px;border-top:5px solid var(--acc);box-shadow:0 6px 16px rgba(27,49,57,.08);padding:22px 20px 18px;}
    .ms-badge{position:relative;width:52px;height:52px;border-radius:14px;background:var(--tint);color:var(--acc);display:inline-flex;align-items:center;justify-content:center;margin-bottom:13px;}
    .ms-badge svg{width:28px;height:28px;}
    .ms-pip{position:absolute;top:-5px;right:-5px;width:22px;height:22px;border-radius:50%;background:#00A972;color:#fff;font-weight:800;font-size:13px;display:inline-flex;align-items:center;justify-content:center;border:3px solid #fff;box-shadow:0 2px 5px rgba(27,49,57,.2);}
    .ms-card h4{margin:0 0 9px;font-size:15.5px;font-weight:800;color:#0b2026;}
    .ms-card ul{list-style:none;margin:0;padding:0;}
    .ms-card li{position:relative;padding-left:18px;font-size:13px;line-height:1.5;color:#3a4d54;margin-bottom:8px;}
    .ms-card li::before{content:'';position:absolute;left:2px;top:7px;width:7px;height:7px;border-radius:2px;transform:rotate(45deg);background:var(--acc);}
    @media(max-width:720px){.ms-grid{grid-template-columns:1fr;}}
  </style>
  <div class="ms-grid">
    <div class="ms-card" style="--acc:#00A972;--tint:rgba(0,169,114,.12);"><div class="ms-badge"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M12 3l7 2.8v5.2c0 4.4-3 7.4-7 8.9-4-1.5-7-4.5-7-8.9V5.8L12 3Z"/><path d="M8.8 11.8l2.2 2.2 4.2-4.4"/></svg><span class="ms-pip">&#10003;</span></div><h4>Production-Grade Serving</h4><ul><li>Highly available, low-latency, scalable serving for small and large workloads.</li></ul></div>
    <div class="ms-card" style="--acc:#00A972;--tint:rgba(0,169,114,.12);"><div class="ms-badge"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M9.5 14.5 14.5 9.5"/><path d="M10.5 6.5 12.3 4.7a4 4 0 015.7 5.7l-1.8 1.8"/><path d="M13.5 17.5l-1.8 1.8a4 4 0 01-5.7-5.7l1.8-1.8"/></svg><span class="ms-pip">&#10003;</span></div><h4>Lakehouse-Unified Serving</h4><ul><li>Automatic feature lookups, monitoring, and unified governance that automate deployment and reduce errors.</li></ul></div>
    <div class="ms-card" style="--acc:#00A972;--tint:rgba(0,169,114,.12);"><div class="ms-badge"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M4 20 14.5 9.5"/><path d="M13.5 8.5l2 2"/><path d="M17.5 2.5l.8 2 2 .8-2 .8-.8 2-.8-2-2-.8 2-.8z"/></svg><span class="ms-pip">&#10003;</span></div><h4>Simplified Deployment</h4><ul><li>Simple, flexible deployment through the UI or API.</li></ul></div>
  </div>
</div>

## D. Real-time Deployment

### A typical real-time model deployment workflow for AI systems
<div class="scn">
  <style>
    .scn { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif; color: #0b2026; width: 100%; max-width: none; margin: 0; padding: 8px 0 16px; }
    .scn * { box-sizing: border-box; }
    .scn .stage { background: transparent; padding: 4px 0; width: 100%; }
    .scn svg { display: block; width: 100%; height: auto; }
    .scn .t { font-size: 14px; fill: #0b2026; }
    .scn .tb { font-size: 15px; font-weight: 700; fill: #0b2026; }
    .scn .tt { font-size: 17px; font-weight: 700; fill: #0b2026; }
    .scn .tw { font-size: 15px; font-weight: 700; fill: #ffffff; }
    .scn .tm { font-size: 14px; fill: #5A6F77; }
    .scn .mono { font-size: 14px; font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; fill: #0b2026; }
    .scn .monod { font-size: 14px; font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; fill: #0b2026; }
    .scn rect.bg { fill: transparent; stroke: #DCE0E2; stroke-width: 1.5; }
    .scn rect.code { fill: #EEEDE9; stroke: #DCE0E2; stroke-width: 1; }
    .scn rect.grp { fill: rgba(255,95,70,0.06); stroke: #FF5F46; stroke-width: 2; stroke-dasharray: 6 5; }
    .scn .flow { stroke: #618794; stroke-width: 2.2; fill: none; marker-end: url(#ar); }
    .scn .flowa { stroke: #FF5F46; stroke-width: 2.4; fill: none; marker-end: url(#arC); }
    .scn .flowg { stroke: #00A972; stroke-width: 2.2; fill: none; marker-end: url(#arG); }
    .scn .cw { display: flex; align-items: center; height: 100%; width: 100%; }
    .scn code.cd { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 14px; color: #0b2026; background: #EEEDE9; border: 1px solid #DCE0E2; border-radius: 6px; padding: 6px 14px; display: block; width: 100%; white-space: nowrap; overflow: hidden; }
    .scn .legend { display: flex; flex-wrap: wrap; gap: 18px; margin-top: 14px; font-size: 14pt; color: #5A6F77; align-items: center; }
    .scn .legend span { display: flex; align-items: center; gap: 8px; }
    .scn .sw { width: 26px; height: 5px; border-radius: 3px; display: inline-block; }
  </style>
  <div class="stage">
    <svg viewBox="0 0 1140 1090" preserveAspectRatio="xMidYMid meet" role="img" aria-label="Databricks custom agent architecture and deployment paths">
      <defs>
        <marker id="ar" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#618794"/></marker>
        <marker id="arC" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#FF5F46"/></marker>
        <marker id="arG" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#00A972"/></marker>
      </defs>
      <!-- ================= ONLINE PATH ================= -->
      <text class="tt" x="16" y="26" fill="#1B5162">1 &nbsp;Online request path</text>
      <!-- clients -->
      <g>
        <rect class="bg" x="330" y="44" width="480" height="52" rx="8"/>
        <rect x="330" y="44" width="480" height="6" rx="3" fill="#618794"/>
        <text class="tb" x="570" y="76" text-anchor="middle">Users &nbsp;·&nbsp; Web App &nbsp;·&nbsp; External API clients</text>
      </g>
      <path class="flow" d="M570,96 L570,140"/>
      <text class="mono" x="584" y="116">HTTPS / REST</text>
      <text class="mono" x="584" y="134" fill="#FF5F46">POST /responses</text>
      <!-- App runtime -->
      <g>
        <rect class="bg" x="90" y="146" width="960" height="196" rx="10"/>
        <rect x="90" y="146" width="960" height="8" rx="4" fill="#FF5F46"/>
        <text class="tt" x="114" y="184">Databricks Apps &nbsp;&mdash;&nbsp; recommended runtime for custom agents</text>
        <text class="tm" x="114" y="206">Git-managed agent code</text>
        <rect x="114" y="220" width="216" height="102" rx="6" fill="rgba(255,95,70,0.10)" stroke="#FF9E94"/>
        <text class="tb" x="128" y="244">Interface</text>
        <text class="mono" x="128" y="266">ResponsesAgent</text>
        <text class="t" x="128" y="288">MLflow AgentServer</text>
        <text class="t" x="128" y="308">or async FastAPI</text>
        <rect x="346" y="220" width="330" height="102" rx="6" fill="rgba(66,153,224,0.10)" stroke="#4299E0"/>
        <text class="tb" x="360" y="244">Application behavior</text>
        <text class="t" x="360" y="266">orchestration · tools · retries</text>
        <text class="t" x="360" y="288">routes · streaming · state</text>
        <text class="t" x="360" y="308">retrieval decisions</text>
        <rect x="692" y="220" width="334" height="102" rx="6" fill="rgba(0,169,114,0.10)" stroke="#00A972"/>
        <text class="tb" x="706" y="244">Framework</text>
        <text class="t" x="706" y="266">OpenAI Agents SDK · LangGraph</text>
        <text class="t" x="706" y="288">LlamaIndex · custom orchestration</text>
        <text class="tm" x="706" y="308">Deployed via DABs</text>
      </g>
      <!-- branches -->
      <path class="flow" d="M330,342 L330,384 L268,384 L268,420"/>
      <text class="mono" x="120" y="376" fill="#4299E0">query / retrieve</text>
      <path class="flow" d="M810,342 L810,384 L866,384 L866,420"/>
      <text class="mono" x="884" y="366" fill="#00A972">generate / embed / call model</text>
      <!-- AI Search -->
      <g>
        <rect class="bg" x="90" y="426" width="440" height="128" rx="10"/>
        <rect x="90" y="426" width="440" height="8" rx="4" fill="#4299E0"/>
        <text class="tt" x="114" y="462">Databricks AI Search index</text>
        <text class="t" x="114" y="490">Low-latency online retrieval</text>
        <text class="tm" x="114" y="518">query text / embedding</text>
        <text class="tm" x="114" y="538">+ ACL &amp; metadata filters</text>
      </g>
      <!-- Model Serving -->
      <g>
        <rect class="bg" x="610" y="426" width="440" height="128" rx="10"/>
        <rect x="610" y="426" width="440" height="8" rx="4" fill="#00A972"/>
        <text class="tt" x="634" y="462">Databricks Model Serving</text>
        <text class="t" x="634" y="490">Foundation Model APIs</text>
        <text class="t" x="634" y="514">Embedding models</text>
        <text class="t" x="634" y="538">Shared custom / managed endpoints</text>
      </g>
      <path class="flow" d="M310,554 L310,596"/>
      <!-- UC -->
      <g>
        <rect class="bg" x="90" y="602" width="440" height="104" rx="10"/>
        <rect x="90" y="602" width="440" height="8" rx="4" fill="#FFAB00"/>
        <text class="tt" x="114" y="640">Governed Unity Catalog managed tables &amp; documents</text>
        <text class="t" x="114" y="666">Unity Catalog permissions,</text>
        <text class="t" x="114" y="690">lineage, and access control</text>
      </g>
      <!-- ================= DEPLOYMENT ================= -->
      <line x1="16" y1="740" x2="1124" y2="740" stroke="#DCE0E2" stroke-width="2"/>
      <text class="tt" x="16" y="778" fill="#1B5162">2 &nbsp;Deployment paths for agent-as-code</text>
      <!-- bundle path -->
      <g>
        <rect class="bg" x="16" y="796" width="546" height="278" rx="10"/>
        <rect x="16" y="796" width="546" height="8" rx="4" fill="#2272B4"/>
        <text class="tb" x="40" y="832" font-size="16">Path A &nbsp;·&nbsp; Apps + DABs</text>
        <rect x="44" y="852" width="212" height="38" rx="6" fill="#1B5162"/>
        <text class="tw" x="150" y="876" text-anchor="middle">Git repository</text>
        <path class="flow" d="M260,871 L296,871"/>
        <rect class="code" x="302" y="852" width="232" height="38" rx="6"/>
        <text class="tb" x="418" y="876" text-anchor="middle">CI/CD or Databricks CLI</text>
        <!-- arrow from CI/CD box down into the dotted coral group -->
        <path class="flowa" d="M418,890 L418,924"/>
        <!-- dotted coral group; code blocks span full inner width -->
        <rect class="grp" x="44" y="928" width="490" height="130" rx="8"/>
        <!-- fallback SVG text (shown only if foreignObject unsupported) -->
        <g class="fb">
          <text class="monod" x="76" y="962">databricks bundle validate</text>
          <text class="monod" x="76" y="998">databricks bundle deploy</text>
          <text class="monod" x="76" y="1034">databricks bundle run</text>
        </g>
        <foreignObject x="60" y="940" width="458" height="34">
          <div xmlns="http://www.w3.org/1999/xhtml" class="cw"><code class="cd">databricks bundle validate</code></div>
        </foreignObject>
        <foreignObject x="60" y="976" width="458" height="34">
          <div xmlns="http://www.w3.org/1999/xhtml" class="cw"><code class="cd">databricks bundle deploy</code></div>
        </foreignObject>
        <foreignObject x="60" y="1012" width="458" height="34">
          <div xmlns="http://www.w3.org/1999/xhtml" class="cw"><code class="cd">databricks bundle run</code></div>
        </foreignObject>
      </g>
      <!-- endpoint path -->
      <g>
        <rect class="bg" x="578" y="796" width="546" height="278" rx="10"/>
        <rect x="578" y="796" width="546" height="8" rx="4" fill="#98102A"/>
        <text class="tb" x="602" y="832" font-size="16">Path B &nbsp;·&nbsp; MLflow artifact &rarr; Serving endpoint</text>
        <g class="fb">
          <text class="monod" x="618" y="876">mlflow.log_model(...) + ResponsesAgent</text>
        </g>
        <foreignObject x="602" y="852" width="500" height="40">
          <div xmlns="http://www.w3.org/1999/xhtml" class="cw"><code class="cd">mlflow.log_model(...) + ResponsesAgent</code></div>
        </foreignObject>
        <path class="flow" d="M851,896 L851,932"/>
        <rect class="code" x="602" y="938" width="498" height="38" rx="6"/>
        <text class="tb" x="618" y="962">Unity Catalog registered model version</text>
        <path class="flow" d="M851,978 L851,1018"/>
        <g class="fb">
          <text class="monod" x="618" y="1046">agents.deploy(...)</text>
        </g>
        <foreignObject x="602" y="1022" width="240" height="40">
          <div xmlns="http://www.w3.org/1999/xhtml" class="cw"><code class="cd">agents.deploy(...)</code></div>
        </foreignObject>
        <path class="flowg" d="M848,1041 L878,1041"/>
        <rect x="884" y="1022" width="216" height="38" rx="6" fill="#00A972"/>
        <text class="tw" x="992" y="1046" text-anchor="middle">Model Serving endpoint</text>
      </g>
    </svg>
  </div>
  <div class="legend">
    <span><i class="sw" style="background:#618794"></i> request / control flow</span>
    <span><i class="sw" style="background:#4299E0"></i> retrieval branch</span>
    <span><i class="sw" style="background:#00A972"></i> model branch</span>
    <span><i class="sw" style="background:#FFAB00"></i> governance layer</span>
  </div>
</div>


## E. Unity Gateway

<div style="max-width: 720px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 700 500" xmlns:xlink="http://www.w3.org/1999/xlink" style="font-family: sans-serif;">
  <defs>
    <marker id="b4arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#6B7780" stroke-width="1.6" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
    <filter id="soft" x="-20%" y="-20%" width="140%" height="140%">
      <feDropShadow dx="0" dy="1.5" stdDeviation="2.5" flood-color="#1B3139" flood-opacity="0.16"/>
    </filter>
  </defs>
    <rect x="260" y="10" width="180" height="52" rx="12" fill="#ffffff" stroke="#E6E8EC" stroke-width="1.2" filter="url(#soft)"/>
  <text x="350" y="34" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">Engineering Budget</text>
  <text x="350" y="50" text-anchor="middle" font-size="11" font-style="italic" fill="#5E7077">$50,000 per month</text>
  <path d="M350 62 L350 92" fill="none" stroke="#6B7780" stroke-width="1.6" stroke-dasharray="5 5" marker-end="url(#b4arrow)"/>
  <rect x="30" y="96" width="640" height="152" rx="16" fill="#ffffff" stroke="#E6E8EC" stroke-width="1.5" filter="url(#soft)"/>
  <text x="368" y="130" text-anchor="middle" font-size="18" font-weight="800" fill="#0b2026">Unity Gateway</text>
  <rect x="43" y="150" width="164" height="32" rx="16" fill="#FF5F46"/>
  <text x="125" y="171" text-anchor="middle" font-size="12.5" font-weight="600" fill="#ffffff">Spend tracking</text>
  <rect x="268" y="150" width="164" height="32" rx="16" fill="#FF5F46"/>
  <text x="350" y="171" text-anchor="middle" font-size="12.5" font-weight="600" fill="#ffffff">Budget policies</text>
  <rect x="493" y="150" width="164" height="32" rx="16" fill="#FF5F46"/>
  <text x="575" y="171" text-anchor="middle" font-size="12.5" font-weight="600" fill="#ffffff">Hard spend caps</text>
  <rect x="43" y="194" width="164" height="32" rx="16" fill="#FF5F46"/>
  <text x="125" y="215" text-anchor="middle" font-size="12.5" font-weight="600" fill="#ffffff">Rate limits</text>
  <rect x="268" y="194" width="164" height="32" rx="16" fill="#FF5F46"/>
  <text x="350" y="215" text-anchor="middle" font-size="12.5" font-weight="600" fill="#ffffff">Chargeback</text>
  <rect x="493" y="194" width="164" height="32" rx="16" fill="#FF5F46"/>
  <text x="575" y="215" text-anchor="middle" font-size="12.5" font-weight="600" fill="#ffffff">Usage attribution</text>
  <path d="M125 248 L125 286" fill="none" stroke="#6B7780" stroke-width="1.6" stroke-dasharray="5 5" marker-end="url(#b4arrow)"/>
  <path d="M350 248 L350 286" fill="none" stroke="#6B7780" stroke-width="1.6" stroke-dasharray="5 5" marker-end="url(#b4arrow)"/>
  <path d="M575 248 L575 286" fill="none" stroke="#6B7780" stroke-width="1.6" stroke-dasharray="5 5" marker-end="url(#b4arrow)"/>
  <rect x="40" y="290" width="170" height="66" rx="12" fill="#ffffff" stroke="#E6E8EC" stroke-width="1.2" filter="url(#soft)"/>
  <foreignObject x="75" y="304" width="100" height="22"><img xmlns="http://www.w3.org/1999/xhtml" src="../Includes/images/icons/cursor.png" style="width:100px;height:22px;object-fit:contain;display:block;"/></foreignObject>
  <text x="125" y="344" text-anchor="middle" font-size="11" fill="#5E7077"><tspan fill="#2F9E6F" font-weight="700">$5.5k</tspan> of $10k used</text>
  <rect x="265" y="290" width="170" height="66" rx="12" fill="#ffffff" stroke="#E6E8EC" stroke-width="1.2" filter="url(#soft)"/>
  <text x="350" y="322" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">Omnigent</text>
  <text x="350" y="344" text-anchor="middle" font-size="11" fill="#5E7077"><tspan fill="#E8A33D" font-weight="700">$22k</tspan> of $25k used</text>
  <rect x="490" y="290" width="170" height="66" rx="12" fill="#ffffff" stroke="#E6E8EC" stroke-width="1.2" filter="url(#soft)"/>
  <text x="575" y="322" text-anchor="middle" font-size="13" font-weight="700" fill="#0b2026">Customer support</text>
  <text x="575" y="344" text-anchor="middle" font-size="11" fill="#5E7077"><tspan fill="#E5484D" font-weight="700">$15k</tspan> of $15k used</text>
  <path d="M85 356 L85 412" fill="none" stroke="#6B7780" stroke-width="1.6" stroke-dasharray="5 5" marker-end="url(#b4arrow)"/>
  <circle cx="85" cy="438" r="24" fill="#ffffff" stroke="#E6E8EC" stroke-width="1.2" filter="url(#soft)"/><foreignObject x="69" y="422" width="32" height="32"><img xmlns="http://www.w3.org/1999/xhtml" src="../Includes/images/icons/openai.png" style="width:32px;height:32px;object-fit:contain;display:block;"/></foreignObject>
  <path d="M165 356 L165 412" fill="none" stroke="#6B7780" stroke-width="1.6" stroke-dasharray="5 5" marker-end="url(#b4arrow)"/>
  <circle cx="165" cy="438" r="24" fill="#ffffff" stroke="#E6E8EC" stroke-width="1.2" filter="url(#soft)"/><foreignObject x="149" y="422" width="32" height="32"><img xmlns="http://www.w3.org/1999/xhtml" src="../Includes/images/icons/claude-anthropic.png" style="width:32px;height:32px;object-fit:contain;display:block;"/></foreignObject>
  <path d="M310 356 L310 412" fill="none" stroke="#6B7780" stroke-width="1.6" stroke-dasharray="5 5" marker-end="url(#b4arrow)"/>
  <circle cx="310" cy="438" r="24" fill="#ffffff" stroke="#E6E8EC" stroke-width="1.2" filter="url(#soft)"/><foreignObject x="294" y="422" width="32" height="32"><img xmlns="http://www.w3.org/1999/xhtml" src="../Includes/images/icons/gemini.png" style="width:32px;height:32px;object-fit:contain;display:block;"/></foreignObject>
  <path d="M390 356 L390 412" fill="none" stroke="#6B7780" stroke-width="1.6" stroke-dasharray="5 5" marker-end="url(#b4arrow)"/>
  <circle cx="390" cy="438" r="24" fill="#ffffff" stroke="#E6E8EC" stroke-width="1.2" filter="url(#soft)"/><foreignObject x="374" y="422" width="32" height="32"><img xmlns="http://www.w3.org/1999/xhtml" src="../Includes/images/icons/genai-brain-icon.png" style="width:32px;height:32px;object-fit:contain;display:block;"/></foreignObject>
  <path d="M535 356 L535 412" fill="none" stroke="#6B7780" stroke-width="1.6" stroke-dasharray="5 5" marker-end="url(#b4arrow)"/>
  <g stroke="#E5484D" stroke-width="2.6" stroke-linecap="round"><line x1="529" y1="378" x2="541" y2="390"/><line x1="529" y1="390" x2="541" y2="378"/></g>
  <circle cx="535" cy="438" r="24" fill="#ffffff" stroke="#E6E8EC" stroke-width="1.2" filter="url(#soft)"/><foreignObject x="519" y="422" width="32" height="32"><img xmlns="http://www.w3.org/1999/xhtml" src="../Includes/images/icons/antigravity.png" style="width:32px;height:32px;object-fit:contain;display:block;"/></foreignObject>
  <path d="M615 356 L615 412" fill="none" stroke="#6B7780" stroke-width="1.6" stroke-dasharray="5 5" marker-end="url(#b4arrow)"/>
  <g stroke="#E5484D" stroke-width="2.6" stroke-linecap="round"><line x1="609" y1="378" x2="621" y2="390"/><line x1="609" y1="390" x2="621" y2="378"/></g>
  <circle cx="615" cy="438" r="24" fill="#ffffff" stroke="#E6E8EC" stroke-width="1.2" filter="url(#soft)"/><clipPath id="gclip615"><rect x="600" y="423" width="30" height="30" rx="4"/></clipPath><foreignObject x="600" y="423" width="80" height="30" clip-path="url(#gclip615)"><img xmlns="http://www.w3.org/1999/xhtml" src="../Includes/images/icons/databricks-genie.png" style="width:80px;height:30px;object-fit:cover;display:block;"/></foreignObject>
  <text x="575" y="487" text-anchor="middle" font-size="13.5" font-style="italic" font-weight="700" fill="#E5484D">Blocked!</text>

</svg>
</div>

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #4299E0); color: white; padding: 12px 18px; cursor: pointer; font-weight: 600; font-size: 12pt; border-radius: 8px; user-select: none;">
How does Unity Gateway control spend?
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 16px 20px; background: #F9F7F4; font-size: 12pt; line-height: 1.7; color: #333;">

<p><strong>Unity Gateway</strong> is the product surface for control. It becomes your single source of truth for AI consumption and spend. Every request from every harness flows through <strong>one governed entry point</strong>, which gives you unified AI tokenomics and spend governance.</p>

<ul style="margin: 12px 0; padding-left: 24px;">
<li>The top box shows the overall <strong>engineering budget</strong>.</li>
<li>The middle layer shows the gateway capabilities: <strong>spend tracking</strong>, <strong>budget policies</strong>, <strong>hard spend caps</strong>, <strong>rate limits</strong>, <strong>chargeback</strong>, and <strong>usage attribution</strong>.</li>
<li>The lower boxes show how that budget can be allocated across teams or tools.</li>
<li>When a bucket reaches its limit, the gateway blocks subsequent requests before the budget is exceeded.</li>
</ul>

</div>
</details>


## F. Features

- **Usage tracking and observability**: Monitor who’s using models and how often to help optimize performance and manage costs.

- **Payload logging**: Capture requests and responses in Unity Catalog, allowing you to debug and improve model quality.

- **Unified Guardrails**: A single source of truth for data policies, with guardrails like PII detection to ensure compliance. 

- **A/B testing and traffic policies**: Fallback/retry policies.

- **Permissions and granular rate limiting**: Only authorized users can access models and APIs, while controlling the number of requests to optimize performance.

## G. Hands-on: Deploy the Model and Collect Feedback

Now that we understand the deployment paradigms and the role of Model Serving and the Unity Gateway, let's deploy the RAG model we registered and evaluated in the previous lessons. We will run the classroom setup, deploy the model with the Agent Bricks Custom Agents, and then use the Review App to interact with it and gather human feedback.

### G1. Classroom Setup

Set up the classroom environment by installing the required libraries and datasets, then run the next cell to display the catalog, schema, and resource names used in this demo.

In [0]:
%run ../Includes/Classroom-Setup-03

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


/home/spark-9ab62f3c-f21d-417d-addb-8a/.ipykernel/307/command-5842534141863397-689038187:9: DeprecationWarning: databricks-vectorsearch is deprecated and has been renamed to databricks-ai-search. Imports under 'databricks.vector_search.*' will continue to work as a thin re-export of 'databricks.ai_search.*', but new code should switch to 'pip install databricks-ai-search' and 'from databricks.ai_search.* import ...'.
  from databricks.vector_search.client import VectorSearchClient



The examples and models presented in this course are intended solely for demonstration and educational purposes.
 Please note that the models and prompt examples may sometimes contain offensive, inaccurate, biased, or harmful content.


### G2. Other Conventions

Throughout this demo, the object `DA` is used. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Please run the code block below to display these details:

In [0]:
model_name = f"{DA.catalog_name}.{DA.schema_name}.getstarted_genai_retrevial_demo"
vs_index_table_name = f"{DA.catalog_name}.{DA.schema_name}.product_embeddings"

print(f"=== Variables that you will need for this demo === \n")
print(f"Catalog Name                : {DA.catalog_name}\n")
print(f"Schema Name                 : {DA.schema_name}\n")
print(f"RAG Registered Model        : {model_name} \n")
print(f"VS index table name         : {vs_index_table_name } \n")

=== Variables that you will need for this demo === 

Catalog Name                : learn_get_started_genai_puli_reddy

Schema Name                 : default

RAG Registered Model        : learn_get_started_genai_puli_reddy.default.getstarted_genai_retrevial_demo 

VS index table name         : learn_get_started_genai_puli_reddy.default.product_embeddings 



### G3. Serve the Model with Agent Bricks Custom Agents

This section demonstrates how to serve a model from the model registry using Databricks Model Serving and the Agent Bricks Custom Agents. When models are served with the Agent Bricks Custom Agents, a Review App is automatically deployed alongside the model, allowing you to interact with the model and gather human feedback on its responses.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">This step is intended for the course instructor only. If you are using your own environment, you may comment out the cells and run them to deploy the model and access the Review App.</p>
</div>
</div>
</div>

In [0]:
import time
import mlflow
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointStateReady, EndpointStateConfigUpdate
from databricks import agents

# Deploy the model with the Agent Bricks Custom Agents, passing VS_INDEX_NAME as an environment variable
deployment_info = agents.deploy(
    model_name, 
    model_version=1,
    scale_to_zero=True,
    environment_vars={"VS_INDEX_NAME": vs_index_table_name}
)

# Wait for the Review App and deployed model to be ready
w = WorkspaceClient()
print("\nWaiting for endpoint to deploy.  This can take 15 - 20 minutes.", end="")

while ((w.serving_endpoints.get(deployment_info.endpoint_name).state.ready == EndpointStateReady.NOT_READY) or (w.serving_endpoints.get(deployment_info.endpoint_name).state.config_update == EndpointStateConfigUpdate.IN_PROGRESS)):
    print(".", end="")
    time.sleep(30)

print("\nThe endpoint is ready!", end="")

Unexpected status resolving file endpoints (403, request ID 05ff4824-0a75-44fc-9aef-6b83e96423b5), using the default endpoints.
/databricks/python/lib/python3.12/site-packages/databricks/agents/deployments.py:641: UserWarning: This endpoint is being deployed without a feedback model, which has been deprecated.
For more information, see: https://docs.databricks.com/aws/en/generative-ai/agent-framework/feedback-model
  warnings.warn(



    Deployment of learn_get_started_genai_puli_reddy.default.getstarted_genai_retrevial_demo version 1 initiated.  This can take up to 15 minutes and the Review App & Query Endpoint will not work until this deployment finishes.

    View status: https://dbc-af847cd2-6ea8.cloud.databricks.com/ml/endpoints/agents_learn_get_started_genai_puli_reddy-default-getstarted_ge/?o=7474653968665155
    Review App: https://dbc-af847cd2-6ea8.cloud.databricks.com/ml/review-v2/7fdec864e16e4159af71e71f7d5dc060/chat?o=7474653968665155

You can refer back to the links above from the endpoint detail page at https://dbc-af847cd2-6ea8.cloud.databricks.com/ml/endpoints/agents_learn_get_started_genai_puli_reddy-default-getstarted_ge/?o=7474653968665155.

To set up monitoring for your deployed agent, see:
https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/production-monitoring

Waiting for endpoint to deploy.  This can take 15 - 20 minutes..................
The endpoint is ready!

### G4. Collect Human Feedback via Review App

The Databricks Review App stages the LLM in an environment where expert stakeholders can engage with it, allowing for conversations, questions, and more. This setup enables the collection of valuable feedback on your application, ensuring the quality and safety of its responses.

Stakeholders can interact with the application bot and provide feedback on these interactions. They can also offer feedback on historical logs, curated traces, or agent outputs.

In [0]:
print(f"Endpoint URL    : {deployment_info.endpoint_url}")
print(f"Review App URL  : {deployment_info.review_app_url}")

Endpoint URL    : https://dbc-af847cd2-6ea8.cloud.databricks.com/ml/endpoints/agents_learn_get_started_genai_puli_reddy-default-getstarted_ge/?o=7474653968665155
Review App URL  : https://dbc-af847cd2-6ea8.cloud.databricks.com/ml/review-v2/7fdec864e16e4159af71e71f7d5dc060/chat?o=7474653968665155


## Conclusion

In this demo, you deployed the registered model using the Agent Bricks Custom Agents, interacted with the deployed model through the Review App, and learned how the Review App can be used to collect human feedback for further analysis and model improvement.

Key takeaways:
- A registered model can be deployed to a real-time serving endpoint using the Agent Bricks Custom Agents.
- Databricks Model Serving exposes the model as an API for downstream applications.
- The Review App lets stakeholders interact with the agent and provide human feedback.
- Collected feedback closes the loop, feeding back into evaluation and model improvement.

This completes the core lifecycle covered in the demos. In the final module you will explore Agent Bricks, and then apply everything end to end in the comprehensive lab.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>

